# MOSPOLI LMS Colab

This notebook starts MOSPOLI LMS inside Google Colab without Docker/QEMU. The default path uses mock embeddings so the UI opens reliably first. A separate cell starts the real Qwen GGUF + llama.cpp stack when GPU/model setup is ready.

In [ ]:
# 1. Clone or update the project.
REPO_URL = "https://github.com/GODIMONGO/MOSPOLI_LMS"
!test -d /content/MOSPOLI_LMS/.git && git -C /content/MOSPOLI_LMS pull --ff-only || git clone "$REPO_URL" /content/MOSPOLI_LMS
%cd /content/MOSPOLI_LMS

In [ ]:
# 2. Install Node.js 22 and build tools.
!apt-get update -y
!apt-get install -y git wget curl ca-certificates cmake build-essential iproute2
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash -
!apt-get install -y nodejs
!node --version
!npm --version

In [ ]:
%%bash
set -euo pipefail
cd /content/MOSPOLI_LMS
mkdir -p deploy/colab .colab/logs

if [ ! -f deploy/colab/serve-frontend.mjs ]; then
cat > deploy/colab/serve-frontend.mjs <<'EOF'
#!/usr/bin/env node
import fs from 'node:fs'
import http from 'node:http'
import path from 'node:path'
import { fileURLToPath } from 'node:url'

const __dirname = path.dirname(fileURLToPath(import.meta.url))
const repoRoot = path.resolve(__dirname, '../..')
const staticDir = path.resolve(process.env.STATIC_DIR ?? path.join(repoRoot, 'dist'))
const port = Number(process.env.FRONTEND_PORT ?? 5173)
const upstream = process.env.AI_NAVIGATION_UPSTREAM ?? 'http://127.0.0.1:3001'
const mimeTypes = new Map([
  ['.css', 'text/css; charset=utf-8'], ['.html', 'text/html; charset=utf-8'],
  ['.js', 'text/javascript; charset=utf-8'], ['.json', 'application/json; charset=utf-8'],
  ['.png', 'image/png'], ['.svg', 'image/svg+xml'], ['.woff', 'font/woff'], ['.woff2', 'font/woff2'],
])

const server = http.createServer(async (request, response) => {
  const url = new URL(request.url ?? '/', `http://${request.headers.host ?? 'localhost'}`)
  if (url.pathname.startsWith('/api/')) {
    const target = new URL(url.pathname + url.search, upstream)
    const proxy = http.request(target, { method: request.method, headers: { ...request.headers, host: target.host } }, (proxyResponse) => {
      response.writeHead(proxyResponse.statusCode ?? 502, proxyResponse.headers)
      proxyResponse.pipe(response)
    })
    proxy.on('error', (error) => {
      response.writeHead(502, { 'Content-Type': 'application/json; charset=utf-8' })
      response.end(JSON.stringify({ error: 'Bad gateway', message: error.message }))
    })
    request.pipe(proxy)
    return
  }

  const safePath = decodeURIComponent(url.pathname === '/' ? '/index.html' : url.pathname)
  let filePath = path.resolve(staticDir, `.${safePath}`)
  if (!filePath.startsWith(staticDir + path.sep)) {
    response.writeHead(403)
    response.end('Forbidden')
    return
  }
  try {
    if (!(await fs.promises.stat(filePath)).isFile()) filePath = path.join(staticDir, 'index.html')
  } catch {
    filePath = path.join(staticDir, 'index.html')
  }
  response.writeHead(200, { 'Content-Type': mimeTypes.get(path.extname(filePath)) ?? 'application/octet-stream' })
  fs.createReadStream(filePath).pipe(response)
})

server.listen(port, '0.0.0.0', () => console.log(`frontend listening on :${port}, proxy /api -> ${upstream}`))
EOF
fi

if [ ! -f deploy/colab/start-colab.sh ]; then
cat > deploy/colab/start-colab.sh <<'EOF'
#!/usr/bin/env bash
set -euo pipefail
REPO_DIR="${REPO_DIR:-/content/MOSPOLI_LMS}"
AI_PORT="${AI_PORT:-3001}"
FRONTEND_PORT="${FRONTEND_PORT:-5173}"
LOG_DIR="${LOG_DIR:-$REPO_DIR/.colab/logs}"
mkdir -p "$LOG_DIR"
cd "$REPO_DIR"

node -e 'const [a,b]=process.versions.node.split(".").map(Number); if (a<20 || (a===20 && b<19)) { console.error(`Node.js ${process.versions.node} is too old; rerun the Node.js install cell`); process.exit(1) }'

if [ -f package-lock.json ]; then npm ci; else npm install; fi
(cd ai-navigation-service && if [ -f package-lock.json ]; then npm ci; else npm install; fi)
npm run build
(cd ai-navigation-service && npm run build)

pkill -f 'ai-navigation-service/dist/server.js' 2>/dev/null || true
pkill -f 'deploy/colab/serve-frontend.mjs' 2>/dev/null || true

nohup env PORT="$AI_PORT" EMBEDDING_MOCK=true ALLOW_EMBEDDING_FALLBACK=true node ai-navigation-service/dist/server.js > "$LOG_DIR/ai-navigation-service.log" 2>&1 &
for i in $(seq 1 60); do curl -fsS "http://127.0.0.1:$AI_PORT/health" >/dev/null 2>&1 && break; sleep 1; done
curl -fsS "http://127.0.0.1:$AI_PORT/health" >/dev/null

nohup env FRONTEND_PORT="$FRONTEND_PORT" AI_NAVIGATION_UPSTREAM="http://127.0.0.1:$AI_PORT" node deploy/colab/serve-frontend.mjs > "$LOG_DIR/frontend.log" 2>&1 &
for i in $(seq 1 60); do curl -fsS "http://127.0.0.1:$FRONTEND_PORT/" >/dev/null 2>&1 && break; sleep 1; done
curl -fsS "http://127.0.0.1:$FRONTEND_PORT/" >/dev/null

echo 'Services are running:'
echo "  ai-navigation-service: http://127.0.0.1:$AI_PORT"
echo "  frontend:              http://127.0.0.1:$FRONTEND_PORT"
echo "  logs:                  $LOG_DIR"
EOF
fi

chmod +x deploy/colab/start-colab.sh
EMBEDDING_MOCK=true ALLOW_EMBEDDING_FALLBACK=true START_TUNNEL=none deploy/colab/start-colab.sh

In [ ]:
# 4. Local checks inside Colab.
%cd /content/MOSPOLI_LMS
!curl -fsS http://127.0.0.1:5173/ | head -n 5
!curl -fsS http://127.0.0.1:3001/health
!curl -fsS -X POST http://127.0.0.1:5173/api/navigation-search -H 'Content-Type: application/json' -d '{"query":"регистрация","locale":"ru"}'

In [ ]:
# 5. Open the site through Colab's built-in authenticated proxy.
# This avoids Cloudflare/localtunnel and usually works even when VPN blocks tunnel providers.
from google.colab import output
from google.colab.output import eval_js

url = eval_js("google.colab.kernel.proxyPort(5173)")
print(url)
output.serve_kernel_port_as_window(5173)

In [ ]:
# 6. Optional: real embeddings with Qwen GGUF + llama.cpp CUDA.
# Use this only after the mock UI works. Runtime -> Change runtime type -> T4 GPU or better.
%cd /content/MOSPOLI_LMS
!nvidia-smi
!EMBEDDING_MOCK=false START_TUNNEL=none deploy/colab/start-colab.sh

In [ ]:
# 7. Diagnostics if port 5173 is refused.
%cd /content/MOSPOLI_LMS
!ps -ef | grep -E 'llama-server|ai-navigation-service/dist/server.js|deploy/colab/serve-frontend.mjs' | grep -v grep || true
!ss -ltnp | grep -E ':(8080|3001|5173)\\b' || true
!tail -n 100 .colab/logs/frontend.log || true
!tail -n 100 .colab/logs/ai-navigation-service.log || true
!tail -n 100 .colab/logs/llama-server.log || true

In [ ]:
# 8. Optional external tunnel. Use only if Colab proxy is not enough.
%cd /content/MOSPOLI_LMS
!wget -q -O /tmp/cloudflared-linux-amd64.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i /tmp/cloudflared-linux-amd64.deb
!cloudflared tunnel --url http://127.0.0.1:5173